[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/terbe2022/RIMS-Archival-Project-/blob/main/notebooks/01_box_inventory.ipynb)

# 01 — Box Inventory

Build the manifest from Box metadata alone. **Nothing is downloaded.**

This is the single most important idea in the pipeline, so it is worth being explicit
about why. Box returns `sha1`, `size`, `name`, `path_collection` and timestamps in the
same API call that lists a folder. That means inventory, deduplication and structural
filtering can all run *before* a single byte is transferred.

On a 100,000-file collection where triage keeps 5%, that is 5,000 downloads instead of
100,000 — and this pass takes minutes rather than hours.

Run `00_environment_check.ipynb` first.


## Setup


In [ ]:
!git clone -q https://github.com/terbe2022/RIMS-Archival-Project-.git 2>/dev/null || true
%cd -q RIMS-Archival-Project- 2>/dev/null || true
!pip install -q box-sdk-gen pandas pyarrow python-dotenv

import os, sys, pandas as pd
sys.path.insert(0,'src')
from google.colab import userdata
os.environ['BOX_DEVELOPER_TOKEN'] = userdata.get('BOX_DEVELOPER_TOKEN')
FOLDER = '318353711369'   # TomHanratty accession


## Crawl

Walks the whole folder tree breadth-first. Checkpoints every 25 folders, so if the
developer token expires mid-run you regenerate it and re-run — it resumes rather than
starting over.

Writing to Drive so the manifest survives the runtime being recycled.


In [ ]:
from box_inventory import crawl, client_from_env, summarise
from pathlib import Path

from google.colab import drive; drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/rims/manifest'); OUT.mkdir(parents=True, exist_ok=True)

df = crawl(client_from_env(), FOLDER, OUT)
print(f'\n{len(df):,} files inventoried')


## What came back

These are the numbers the whole plan has been assuming. Post them on the GitHub issue.


In [ ]:
summarise(df)


## The two numbers that matter most

**Duplicate rate.** The design assumes 40–70% of a collection falls out for free.
That is a guess borrowed from typical personal drives, not something measured here.
If this comes back at 8%, the triage math changes and the design doc needs updating.

**Extension distribution.** The format gap analysis was built from a curated Box corpus,
not a researcher's accession. If formats appear here that POC 2 never handled — GIS,
instrument data, CAD, statistical packages — those lanes move up the priority order.


In [ ]:
total = len(df)
dupes = total - df['box_sha1'].nunique(dropna=True)
print(f'exact duplicates: {dupes:,} of {total:,}  ({dupes/total*100:.1f}%)')
print(f'design assumed:   40-70% falls out at zero cost')
print()
print('extensions POC 2 never handled:')
handled = {'pdf','doc','docx','xls','xlsx','ppt','pptx','html','htm','rtf','txt',
           'jpg','jpeg','png','gif','tif','tiff','eps','zip','mhtml','mht'}
unhandled = df[~df['extension'].isin(handled)]['extension'].value_counts()
print(unhandled.head(30).to_string())


## Folder shape

Folder structure carries meaning — a folder of mixed formats is usually one project, and
the individually unimpressive files in it inherit that significance. Worth seeing how
deep and how wide this collection actually is before designing the folder-weighting.


In [ ]:
df['depth'] = df['box_path'].str.count('/')
print('files per folder depth:')
print(df['depth'].value_counts().sort_index().to_string())
print(f'\ndeepest path: {df["depth"].max()} levels')
print(f'distinct folders: {df["box_folder_id"].nunique():,}')
print(f'median files per folder: {df.groupby("box_folder_id").size().median():.0f}')


## Size distribution

Tells us where the processing cost will actually land. A handful of very large files can
dominate wall-clock time even when they are a rounding error in the file count.


In [ ]:
s = df['size_bytes'].fillna(0)
print(f'total:  {s.sum()/1e9:,.1f} GB')
print(f'median: {s.median()/1e6:,.2f} MB')
print(f'mean:   {s.mean()/1e6:,.2f} MB')
print(f'largest 10 files hold {s.nlargest(10).sum()/s.sum()*100:.1f}% of all bytes')
print()
print(df.nlargest(10,'size_bytes')[['name','extension','size_bytes']].to_string(index=False))


---

## Before you leave

1. Post the summary numbers as a comment on the inventory issue
2. Edit → **Clear all outputs** — this notebook will have printed real filenames
3. Save a copy to GitHub, on a branch, then open a pull request

Point 2 matters here more than usual: the outputs above contain actual file and folder
names from a real accession. Those should not land in a public repo.
